### 입출력 오류 재현과 진단
- 파일 : n1-5_io_errors.ipynb
- 역할 : 잘못된 전처리(배치, 채널, 정규화)를 의도적으로 만들어 배포 오류의 증상을 재현하고 진단 순서를 익힙니다.
- 기능 :
    1. 기준 전처리와 Top-1 도우미 함수  
    2. 차원 순서를 일부러 NHWC로 넣어 보기  
    3. 채널 순서를 일부러 BGR로 넣어 보기  
    4. Normalize를 일부러 빠뜨려 보기  
    5. dtype을 일부러 float64로 넣어 보기  


>> [준비] 실습 저장소 받기 (노트북 전용 셀)
<hr>

In [ ]:
#- 실습 저장소 받기: 노트북마다 런타임이 분리되므로 새 세션에서 한 번 실행
import os
if not os.path.exists("/content/npu-course"):
    !git clone -q https://github.com/npu-ondevice-ai/npu-course.git /content/npu-course
%cd /content/npu-course

# 확인 포인트 ------------------------------------------
# - 현재 위치 /content/npu-course 출력 체크

>> [준비 2] 앞 노트북 산출물 준비 (노트북 전용 셀)

In [ ]:
#- ONNX 변환과 검증 노트북의 산출물(sess, W_MNET, img)을 이 세션에 준비
!pip install -q onnx onnxruntime "protobuf<6"
import os
import numpy as np
import torch
import onnxruntime as ort
from torchvision import models
from PIL import Image

#- ONNX 파일: 저장소 models/에 있으면 사용, 없으면 즉석에서 변환
W_MNET = models.MobileNet_V2_Weights.IMAGENET1K_V1
ONNX = "models/mobilenet_v2.onnx"
if not os.path.exists(ONNX):
    m = models.mobilenet_v2(weights=W_MNET).eval()
    torch.onnx.export(m, torch.randn(1, 3, 224, 224), ONNX, opset_version=13,
                      input_names=["input"], output_names=["output"], dynamo=False)

#- 검증 실습이 이어받는 객체들
sess = ort.InferenceSession(ONNX, providers=["CPUExecutionProvider"])
img = Image.open("data/sample.jpg")

# 확인 포인트 ------------------------------------------
# - 오류 없이 완료 (sess와 img 준비)

>> [코드 3-10] 기준 전처리와 Top-1 도우미 함수 (n1-5_io_errors.ipynb, STEP 1)
<hr>

In [ ]:
#- 모듈로딩
from PIL import Image

#- 파일 상수: 실습 입력 이미지
IMG_FILE = "data/sample.jpg"
W_MNET   = models.MobileNet_V2_Weights.IMAGENET1K_V1

#- 기준 전처리 상수와 클래스 이름
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
categories = W_MNET.meta["categories"]

#- 올바른 전처리 (추론 파이프라인과 동일)
def preprocess_correct(img):
    #- 추론 파이프라인과 동일한 전처리
    x = np.asarray(img.convert("RGB").resize((224, 224))).astype(np.float32) / 255.0
    x = (x - MEAN) / STD
    return x.transpose(2, 0, 1)[None].astype(np.float32)            # (1, 3, 224, 224)

#- Top-1 클래스와 확신도를 돌려주는 도우미
def top1(x):
    logits = sess.run(None, {"input": x})[0][0]  # 추론 실행, 로짓 1,000개 추출
    p = np.exp(logits - logits.max())            # 최댓값을 빼는 안정적인 Softmax
    p /= p.sum()                                 # 합이 1이 되도록 정규화
    i = int(p.argmax())                          # 확률이 가장 큰 클래스 번호
    return categories[i], float(p[i])            # (클래스 이름, 확신도)

#- 특정 이미지의 Top-1 클래스 분류
img = Image.open(IMG_FILE).convert("RGB")
label, prob = top1(preprocess_correct(img))
print(f"기준 결과: {label} ({prob*100:.1f}%)")

# 확인 포인트 --------------------------------------------------------------------------
# - 추론 파이프라인에서 같은 이미지로 얻은 Top-1과 동일 클래스 출력 체크

>> [코드 3-11] 차원 순서를 일부러 NHWC로 넣어 보기 (STEP 2)
<hr>

In [ ]:
#- 올바른 입력과 일부러 차원 순서를 바꾼 입력
x_correct = preprocess_correct(img)              # (1, 3, 224, 224)
x_nhwc = x_correct.transpose(0, 2, 3, 1)         # 일부러 (1, 224, 224, 3)로

#- NHWC 입력으로 추론 시도
try:
    sess.run(None, {"input": x_nhwc})
except Exception as e:
    print(f"오류 발생: {type(e).__name__}")

# 확인 포인트 -------------------------------------------------------------------------
# - 입력 shape 불일치 오류 출력 체크

>> [코드 3-12] 채널 순서를 일부러 BGR로 넣어 보기 (STEP 3)
<hr>

In [ ]:
#- 채널 순서를 일부러 BGR로 뒤집은 이미지
img_bgr = Image.fromarray(np.asarray(img)[:, :, ::-1])   # 채널 반전
label, prob = top1(preprocess_correct(img_bgr))

print(f"BGR 입력 결과: {label} ({prob*100:.1f}%)")

# 확인 포인트 --------------------------------------------------------------------------
# - 오류 없이 실행, 기준과 다른 클래스 또는 급락한 확신도 확인

>> [코드 3-13] Normalize를 일부러 빠뜨려 보기 (STEP 4)
<hr>

In [ ]:
#- /255와 mean/std를 일부러 건너뛴 입력
x_raw = np.asarray(img.resize((224,224))).astype(np.float32).transpose(2, 0, 1)[None]
label, prob = top1(x_raw)                                           # 0~255 값 그대로

print(f"Normalize 누락 결과: {label} ({prob*100:.1f}%)")

# 확인 포인트 -------------------------------------------------------------------------
# - 오류 없이 실행, 무관한 클래스의 높은 확신도 확인

>> [코드 3-14] dtype을 일부러 float64로 넣어 보기 (STEP 5)
<hr>

In [ ]:
#- dtype을 일부러 float64로 바꾼 입력
x_f64 = preprocess_correct(img).astype(np.float64)

#- float64 입력으로 추론 시도
try:
    sess.run(None, {"input": x_f64})
except Exception as e:
    print(f"오류 발생: {type(e).__name__}")

# 확인 포인트 -------------------------------------------------------------------------
# - float(float32) 기대와 double(float64) 입력의 형 불일치 오류 확인